# 🔬 Clusterização de Síndromes

**Project:** ProEpi - Guardians of Health
**Author:** Danielly Xavier  
**Email:** danielly.xavier@outlook.com  
**Date:** September 2024

## 🎯 Objetivos
Aplicar diferentes algoritmos de clusterização para identificar padrões e agrupar síndromes baseadas nos sintomas relatados pelos usuários. Utilizar algoritmos apropriados para dados categóricos.

## 📋 Estrutura do Notebook
1. **Configuração do Ambiente**
2. **Carregamento e Preparação dos Dados**
3. **Extração e Processamento de Sintomas**
4. **Criação da Matriz de Features**
5. **Clusterização com K-Modes**
6. **Clusterização com K-Means (após encoding)**
7. **Clusterização Hierárquica**
8. **Clusterização com DBSCAN**
9. **Análise Comparativa e Visualizações**
10. **Resumo e Conclusões**


## 1. Configuração do Ambiente


In [1]:
# Essential imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import re
from collections import Counter
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist, squareform
import matplotlib.patches as mpatches

# K-Modes para dados categóricos
try:
    from kmodes.kmodes import KModes
    KMODES_AVAILABLE = True
except ImportError:
    print("⚠️ kmodes não instalado. Instale com: pip install kmodes")
    KMODES_AVAILABLE = False

# Gower Distance
try:
    import gower
    GOWER_AVAILABLE = True
except ImportError:
    print("⚠️ gower não instalado. Instale com: pip install gower")
    GOWER_AVAILABLE = False

# Visualization settings
plt.style.use('default')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

# Pandas settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("✅ Ambiente configurado com sucesso!")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"K-Modes disponível: {KMODES_AVAILABLE}")
print(f"Gower disponível: {GOWER_AVAILABLE}")


⚠️ gower não instalado. Instale com: pip install gower
✅ Ambiente configurado com sucesso!
Pandas: 2.3.2
NumPy: 2.3.3
K-Modes disponível: True
Gower disponível: False


## 2. Carregamento e Preparação dos Dados


In [2]:
def to_snake_case(name):
    """
    Converte um nome de coluna para o padrão snake_case do Python
    """
    if pd.isna(name) or not str(name).strip():
        return None
    
    name = str(name).strip()
    # Remove caracteres especiais e espaços, substitui por underscore
    name = re.sub(r'[^\w\s]', '_', name)
    # Substitui espaços por underscore
    name = re.sub(r'\s+', '_', name)
    # Converte para minúsculas
    name = name.lower()
    # Remove múltiplos underscores
    name = re.sub(r'_+', '_', name)
    # Remove underscores no início e fim
    name = name.strip('_')
    
    return name

# Carregar dados
csv_path = '../data/raw/tmp/reports-gds-unb-2022-2024.csv'
print(f"📁 Carregando arquivo CSV: {csv_path}")

df = pd.read_csv(csv_path, header=0)
print(f"✅ CSV carregado: {df.shape[0]:,} linhas, {df.shape[1]} colunas")

# Converter nomes de colunas para snake_case
df.columns = [to_snake_case(col) if col else f'col_{i}' for i, col in enumerate(df.columns)]
print(f"✅ Colunas convertidas para snake_case")

# Filtrar apenas registros com sintomas
df_symptoms = df[df['report_symptoms'].notna()].copy()
print(f"\n📊 Registros com sintomas: {len(df_symptoms):,} ({len(df_symptoms)/len(df)*100:.2f}%)")

print(f"\n✅ Dados carregados e preparados!")


📁 Carregando arquivo CSV: ../data/raw/tmp/reports-gds-unb-2022-2024.csv
✅ CSV carregado: 2,067,080 linhas, 17 colunas
✅ Colunas convertidas para snake_case

📊 Registros com sintomas: 21,061 (1.02%)

✅ Dados carregados e preparados!


## 3. Extração e Processamento de Sintomas


In [3]:
def extract_symptoms(symptom_string):
    """
    Extrai sintomas individuais de uma string de sintomas
    """
    if pd.isna(symptom_string) or not str(symptom_string).strip():
        return []
    
    symptom_str = str(symptom_string).strip()
    
    # Dividir por linhas
    symptoms = symptom_str.split('\n')
    
    # Limpar e normalizar sintomas
    cleaned_symptoms = []
    for symptom in symptoms:
        symptom = symptom.strip()
        # Remover prefixos comuns como "- ", "• ", etc.
        symptom = re.sub(r'^[-•\s]+', '', symptom)
        # Remover valores vazios e separadores
        if symptom and symptom not in ['---', '-', '']:
            # Normalizar: remover espaços extras e converter para minúsculas
            symptom = re.sub(r'\s+', ' ', symptom).strip().lower()
            if symptom:
                cleaned_symptoms.append(symptom)
    
    return cleaned_symptoms

# Extrair sintomas de cada registro
print("🔄 Extraindo sintomas de cada registro...")
df_symptoms['symptoms_list'] = df_symptoms['report_symptoms'].apply(extract_symptoms)

# Filtrar registros que têm pelo menos um sintoma
df_symptoms = df_symptoms[df_symptoms['symptoms_list'].apply(len) > 0].copy()
print(f"✅ Registros com sintomas válidos: {len(df_symptoms):,}")

# Coletar todos os sintomas únicos
all_symptoms = []
for symptoms in df_symptoms['symptoms_list']:
    all_symptoms.extend(symptoms)

symptom_counter = Counter(all_symptoms)
unique_symptoms = sorted(symptom_counter.keys())

print(f"\n📊 Estatísticas de Sintomas:")
print(f"   • Total de sintomas únicos: {len(unique_symptoms)}")
print(f"   • Total de ocorrências de sintomas: {len(all_symptoms):,}")
print(f"   • Média de sintomas por registro: {np.mean([len(s) for s in df_symptoms['symptoms_list']]):.2f}")

# Mostrar top 20 sintomas mais frequentes
print(f"\n🔝 Top 20 Sintomas Mais Frequentes:")
for i, (symptom, count) in enumerate(symptom_counter.most_common(20), 1):
    percentage = (count / len(df_symptoms)) * 100
    print(f"   {i:2d}. {symptom[:50]:50s} : {count:5,} ({percentage:5.2f}%)")


🔄 Extraindo sintomas de cada registro...
✅ Registros com sintomas válidos: 21,061

📊 Estatísticas de Sintomas:
   • Total de sintomas únicos: 60
   • Total de ocorrências de sintomas: 73,201
   • Média de sintomas por registro: 3.48

🔝 Top 20 Sintomas Mais Frequentes:
    1. cansaco                                            : 8,322 (39.51%)
    2. congestãonasal                                     : 8,193 (38.90%)
    3. tosse                                              : 7,651 (36.33%)
    4. dordegarganta                                      : 7,485 (35.54%)
    5. dorcabeca                                          : 6,898 (32.75%)
    6. mal-estar                                          : 5,367 (25.48%)
    7. coriza                                             : 2,497 (11.86%)
    8. febre                                              : 2,380 (11.30%)
    9. calafrios                                          : 1,850 ( 8.78%)
   10. dornosmúsculos                                   

## 4. Criação da Matriz de Features


In [4]:
# Filtrar sintomas que aparecem em pelo menos N registros (para reduzir dimensionalidade)
MIN_OCCURRENCES = 10  # Sintomas que aparecem em pelo menos 10 registros
frequent_symptoms = [symptom for symptom, count in symptom_counter.items() 
                      if count >= MIN_OCCURRENCES]

print(f"📊 Sintomas frequentes (≥{MIN_OCCURRENCES} ocorrências): {len(frequent_symptoms)}")
print(f"   • Redução de {len(unique_symptoms)} para {len(frequent_symptoms)} sintomas")
print(f"   • Cobertura: {sum(symptom_counter[s] for s in frequent_symptoms) / len(all_symptoms) * 100:.2f}% das ocorrências")

# Criar matriz binária (one-hot encoding)
print("\n🔄 Criando matriz binária de sintomas...")
symptom_matrix = np.zeros((len(df_symptoms), len(frequent_symptoms)), dtype=int)

for idx, symptoms_list in enumerate(df_symptoms['symptoms_list']):
    for symptom in symptoms_list:
        if symptom in frequent_symptoms:
            symptom_idx = frequent_symptoms.index(symptom)
            symptom_matrix[idx, symptom_idx] = 1

# Converter para DataFrame para facilitar análise
df_symptom_matrix = pd.DataFrame(symptom_matrix, columns=frequent_symptoms, index=df_symptoms.index)

print(f"✅ Matriz criada: {symptom_matrix.shape[0]:,} registros × {symptom_matrix.shape[1]} sintomas")
print(f"   • Densidade da matriz: {(symptom_matrix.sum() / symptom_matrix.size) * 100:.2f}%")

# Estatísticas da matriz
print(f"\n📊 Estatísticas da Matriz:")
print(f"   • Média de sintomas por registro: {symptom_matrix.sum(axis=1).mean():.2f}")
print(f"   • Desvio padrão: {symptom_matrix.sum(axis=1).std():.2f}")
print(f"   • Mínimo: {symptom_matrix.sum(axis=1).min()}")
print(f"   • Máximo: {symptom_matrix.sum(axis=1).max()}")


📊 Sintomas frequentes (≥10 ocorrências): 52
   • Redução de 60 para 52 sintomas
   • Cobertura: 99.94% das ocorrências

🔄 Criando matriz binária de sintomas...
✅ Matriz criada: 21,061 registros × 52 sintomas
   • Densidade da matriz: 6.68%

📊 Estatísticas da Matriz:
   • Média de sintomas por registro: 3.47
   • Desvio padrão: 2.82
   • Mínimo: 0
   • Máximo: 39


## 5. Clusterização com K-Modes


In [ ]:
if KMODES_AVAILABLE:
    print("🔬 Aplicando K-Modes (algoritmo específico para dados categóricos)...")
    
    # K-Modes funciona melhor com dados categóricos representados como strings
    # Converter matriz binária para formato categórico
    symptom_data_categorical = []
    for row in symptom_matrix:
        # Criar lista de sintomas presentes (índices)
        present_symptoms = [str(i) for i, val in enumerate(row) if val == 1]
        symptom_data_categorical.append(present_symptoms)
    
    # Determinar número ótimo de clusters usando elbow method
    print("\n📊 Determinando número ótimo de clusters (Elbow Method)...")
    max_clusters = min(15, len(df_symptoms) // 10)  # Limitar a um número razoável
    cost_values = []
    k_range = range(2, max_clusters + 1)
    
    for k in k_range:
        try:
            kmodes = KModes(n_clusters=k, init='Huang', n_init=5, verbose=0, random_state=42)
            clusters = kmodes.fit_predict(symptom_matrix)
            cost_values.append(kmodes.cost_)
            print(f"   K={k}: Cost={kmodes.cost_:.2f}")
        except Exception as e:
            print(f"   K={k}: Erro - {e}")
            cost_values.append(np.inf)
    
    # Visualizar elbow curve
    plt.figure(figsize=(10, 6))
    plt.plot(list(k_range), cost_values, marker='o', linewidth=2, markersize=8)
    plt.xlabel('Número de Clusters (K)', fontsize=12)
    plt.ylabel('Cost (Within-cluster sum of distances)', fontsize=12)
    plt.title('K-Modes: Elbow Method para Determinar K Ótimo', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Escolher K baseado no elbow (pode ser ajustado manualmente)
    optimal_k = 5  # Valor padrão, pode ser ajustado baseado no gráfico
    print(f"\n✅ Usando K={optimal_k} clusters")
    
    # Aplicar K-Modes
    kmodes = KModes(n_clusters=optimal_k, init='Huang', n_init=10, verbose=1, random_state=42)
    clusters_kmodes = kmodes.fit_predict(symptom_matrix)
    
    # Adicionar clusters ao DataFrame
    df_symptoms['cluster_kmodes'] = clusters_kmodes
    
    # Análise dos clusters
    print(f"\n📊 Análise dos Clusters (K-Modes):")
    cluster_counts = pd.Series(clusters_kmodes).value_counts().sort_index()
    for cluster_id, count in cluster_counts.items():
        percentage = (count / len(df_symptoms)) * 100
        print(f"   Cluster {cluster_id}: {count:5,} registros ({percentage:5.2f}%)")
    
    # Identificar sintomas mais comuns em cada cluster
    print(f"\n🔍 Sintomas Característicos por Cluster:")
    for cluster_id in sorted(cluster_counts.index):
        cluster_mask = clusters_kmodes == cluster_id
        cluster_symptoms = symptom_matrix[cluster_mask]
        symptom_freq = cluster_symptoms.sum(axis=0)
        
        # Top 5 sintomas mais frequentes neste cluster
        top_symptoms_idx = np.argsort(symptom_freq)[-5:][::-1]
        top_symptoms = [(frequent_symptoms[idx], symptom_freq[idx], 
                        (symptom_freq[idx] / cluster_symptoms.shape[0]) * 100) 
                       for idx in top_symptoms_idx if symptom_freq[idx] > 0]
        
        print(f"\n   Cluster {cluster_id} ({cluster_counts[cluster_id]:,} registros):")
        for symptom, count, pct in top_symptoms:
            print(f"      • {symptom[:45]:45s} : {count:3d} ({pct:5.1f}%)")
    
    print("\n✅ K-Modes concluído!")
else:
    print("❌ K-Modes não disponível. Instale com: pip install kmodes")


🔬 Aplicando K-Modes (algoritmo específico para dados categóricos)...

📊 Determinando número ótimo de clusters (Elbow Method)...


## 6. Clusterização com K-Means


In [ ]:
print("🔬 Aplicando K-Means (após encoding binário)...")

# K-Means funciona com dados numéricos (nossa matriz já é binária)
# Normalizar os dados pode ajudar
scaler = StandardScaler()
symptom_matrix_scaled = scaler.fit_transform(symptom_matrix)

# Determinar número ótimo de clusters usando elbow method e silhouette score
print("\n📊 Determinando número ótimo de clusters...")
max_clusters = min(15, len(df_symptoms) // 10)
inertias = []
silhouette_scores = []
k_range = range(2, max_clusters + 1)

for k in k_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42, max_iter=300)
    clusters = kmeans.fit_predict(symptom_matrix_scaled)
    inertias.append(kmeans.inertia_)
    
    # Calcular silhouette score
    if len(np.unique(clusters)) > 1:
        sil_score = silhouette_score(symptom_matrix_scaled, clusters)
        silhouette_scores.append(sil_score)
    else:
        silhouette_scores.append(-1)
    
    print(f"   K={k}: Inertia={kmeans.inertia_:.2f}, Silhouette={silhouette_scores[-1]:.3f}")

# Visualizar resultados
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow curve
axes[0].plot(list(k_range), inertias, marker='o', linewidth=2, markersize=8, color='#2E86AB')
axes[0].set_xlabel('Número de Clusters (K)', fontsize=12)
axes[0].set_ylabel('Inertia (Within-cluster sum of squares)', fontsize=12)
axes[0].set_title('K-Means: Elbow Method', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Silhouette scores
axes[1].plot(list(k_range), silhouette_scores, marker='s', linewidth=2, markersize=8, color='#A23B72')
axes[1].set_xlabel('Número de Clusters (K)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('K-Means: Silhouette Score', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Escolher K baseado no silhouette score
optimal_k_kmeans = list(k_range)[np.argmax(silhouette_scores)]
print(f"\n✅ K ótimo baseado em Silhouette Score: {optimal_k_kmeans}")

# Aplicar K-Means com K ótimo
kmeans = KMeans(n_clusters=optimal_k_kmeans, init='k-means++', n_init=10, 
                random_state=42, max_iter=300)
clusters_kmeans = kmeans.fit_predict(symptom_matrix_scaled)

# Adicionar clusters ao DataFrame
df_symptoms['cluster_kmeans'] = clusters_kmeans

# Calcular métricas
silhouette_avg = silhouette_score(symptom_matrix_scaled, clusters_kmeans)
print(f"\n📊 Métricas K-Means:")
print(f"   • Silhouette Score: {silhouette_avg:.3f}")

# Análise dos clusters
print(f"\n📊 Análise dos Clusters (K-Means):")
cluster_counts = pd.Series(clusters_kmeans).value_counts().sort_index()
for cluster_id, count in cluster_counts.items():
    percentage = (count / len(df_symptoms)) * 100
    print(f"   Cluster {cluster_id}: {count:5,} registros ({percentage:5.2f}%)")

# Identificar sintomas mais comuns em cada cluster
print(f"\n🔍 Sintomas Característicos por Cluster:")
for cluster_id in sorted(cluster_counts.index):
    cluster_mask = clusters_kmeans == cluster_id
    cluster_symptoms = symptom_matrix[cluster_mask]
    symptom_freq = cluster_symptoms.sum(axis=0)
    
    # Top 5 sintomas mais frequentes neste cluster
    top_symptoms_idx = np.argsort(symptom_freq)[-5:][::-1]
    top_symptoms = [(frequent_symptoms[idx], symptom_freq[idx], 
                    (symptom_freq[idx] / cluster_symptoms.shape[0]) * 100) 
                   for idx in top_symptoms_idx if symptom_freq[idx] > 0]
    
    print(f"\n   Cluster {cluster_id} ({cluster_counts[cluster_id]:,} registros):")
    for symptom, count, pct in top_symptoms:
        print(f"      • {symptom[:45]:45s} : {count:3d} ({pct:5.1f}%)")

print("\n✅ K-Means concluído!")


## 7. Clusterização Hierárquica


In [ ]:
print("🔬 Aplicando Clusterização Hierárquica...")

# Para dados binários, usar distância de Jaccard ou Hamming
# Usaremos distância de Hamming (proporção de features diferentes)
print("\n📊 Calculando matriz de distâncias (Hamming)...")

# Para grandes datasets, calcular distância apenas para uma amostra
sample_size = min(1000, len(df_symptoms))
if len(df_symptoms) > sample_size:
    print(f"   ⚠️ Dataset grande ({len(df_symptoms):,} registros). Usando amostra de {sample_size} registros.")
    sample_indices = np.random.choice(len(df_symptoms), sample_size, replace=False)
    sample_matrix = symptom_matrix[sample_indices]
    sample_df = df_symptoms.iloc[sample_indices].copy()
else:
    sample_matrix = symptom_matrix
    sample_df = df_symptoms.copy()
    sample_indices = np.arange(len(df_symptoms))

# Calcular distância de Hamming (para dados binários)
# Hamming distance = número de posições onde os valores diferem
from scipy.spatial.distance import hamming

# Calcular linkage matrix
print("   Calculando linkage matrix...")
linkage_matrix = linkage(sample_matrix, method='ward', metric='euclidean')

# Visualizar dendrograma
print("\n📊 Gerando dendrograma...")
plt.figure(figsize=(15, 8))
dendrogram(linkage_matrix, 
           truncate_mode='level', 
           p=10,  # Mostrar apenas os últimos 10 níveis
           leaf_font_size=10,
           color_threshold=0.7*max(linkage_matrix[:,2]))
plt.title('Dendrograma - Clusterização Hierárquica', fontsize=14, fontweight='bold')
plt.xlabel('Índice do Registro ou Cluster', fontsize=12)
plt.ylabel('Distância', fontsize=12)
plt.tight_layout()
plt.show()

# Aplicar clusterização hierárquica com número de clusters
optimal_k_hierarchical = 5  # Pode ser ajustado baseado no dendrograma
print(f"\n✅ Aplicando clusterização hierárquica com {optimal_k_hierarchical} clusters...")

hierarchical = AgglomerativeClustering(n_clusters=optimal_k_hierarchical, linkage='ward')
clusters_hierarchical = hierarchical.fit_predict(sample_matrix)

# Adicionar clusters ao DataFrame da amostra
sample_df['cluster_hierarchical'] = clusters_hierarchical

# Aplicar ao dataset completo usando o modelo treinado
# Para o dataset completo, precisamos recalcular
print("   Aplicando ao dataset completo...")
hierarchical_full = AgglomerativeClustering(n_clusters=optimal_k_hierarchical, linkage='ward')
clusters_hierarchical_full = hierarchical_full.fit_predict(symptom_matrix)
df_symptoms['cluster_hierarchical'] = clusters_hierarchical_full

# Calcular métricas
silhouette_avg = silhouette_score(symptom_matrix, clusters_hierarchical_full)
print(f"\n📊 Métricas Clusterização Hierárquica:")
print(f"   • Silhouette Score: {silhouette_avg:.3f}")

# Análise dos clusters
print(f"\n📊 Análise dos Clusters (Hierárquica):")
cluster_counts = pd.Series(clusters_hierarchical_full).value_counts().sort_index()
for cluster_id, count in cluster_counts.items():
    percentage = (count / len(df_symptoms)) * 100
    print(f"   Cluster {cluster_id}: {count:5,} registros ({percentage:5.2f}%)")

# Identificar sintomas mais comuns em cada cluster
print(f"\n🔍 Sintomas Característicos por Cluster:")
for cluster_id in sorted(cluster_counts.index):
    cluster_mask = clusters_hierarchical_full == cluster_id
    cluster_symptoms = symptom_matrix[cluster_mask]
    symptom_freq = cluster_symptoms.sum(axis=0)
    
    # Top 5 sintomas mais frequentes neste cluster
    top_symptoms_idx = np.argsort(symptom_freq)[-5:][::-1]
    top_symptoms = [(frequent_symptoms[idx], symptom_freq[idx], 
                    (symptom_freq[idx] / cluster_symptoms.shape[0]) * 100) 
                   for idx in top_symptoms_idx if symptom_freq[idx] > 0]
    
    print(f"\n   Cluster {cluster_id} ({cluster_counts[cluster_id]:,} registros):")
    for symptom, count, pct in top_symptoms:
        print(f"      • {symptom[:45]:45s} : {count:3d} ({pct:5.1f}%)")

print("\n✅ Clusterização Hierárquica concluída!")


## 8. Clusterização com DBSCAN


In [ ]:
print("🔬 Aplicando DBSCAN (Density-Based Spatial Clustering)...")

# DBSCAN é útil para encontrar clusters de forma arbitrária e identificar outliers
# Usar distância de Hamming para dados binários
print("\n📊 Testando diferentes parâmetros de DBSCAN...")

# Testar diferentes valores de eps e min_samples
eps_values = [0.3, 0.5, 0.7, 1.0, 1.5]
min_samples_values = [5, 10, 15, 20]

best_eps = None
best_min_samples = None
best_n_clusters = 0
best_silhouette = -1

results_dbscan = []

for eps in eps_values:
    for min_samples in min_samples_values:
        dbscan = DBSCAN(eps=eps, min_samples=min_samples, metric='hamming')
        clusters = dbscan.fit_predict(symptom_matrix)
        
        n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
        n_noise = list(clusters).count(-1)
        
        # Calcular silhouette apenas se houver mais de 1 cluster e não for tudo ruído
        if n_clusters > 1 and n_noise < len(clusters) * 0.5:
            # Filtrar ruído para cálculo de silhouette
            mask = clusters != -1
            if mask.sum() > 1 and len(np.unique(clusters[mask])) > 1:
                sil_score = silhouette_score(symptom_matrix[mask], clusters[mask])
            else:
                sil_score = -1
        else:
            sil_score = -1
        
        results_dbscan.append({
            'eps': eps,
            'min_samples': min_samples,
            'n_clusters': n_clusters,
            'n_noise': n_noise,
            'silhouette': sil_score
        })
        
        if sil_score > best_silhouette and n_clusters >= 2:
            best_silhouette = sil_score
            best_eps = eps
            best_min_samples = min_samples
            best_n_clusters = n_clusters

# Mostrar resultados
df_dbscan_results = pd.DataFrame(results_dbscan)
print("\n📊 Top 10 Configurações de DBSCAN:")
print(df_dbscan_results.nlargest(10, 'silhouette')[['eps', 'min_samples', 'n_clusters', 'n_noise', 'silhouette']])

# Aplicar DBSCAN com melhores parâmetros
if best_eps is not None:
    print(f"\n✅ Aplicando DBSCAN com eps={best_eps}, min_samples={best_min_samples}...")
    dbscan = DBSCAN(eps=best_eps, min_samples=best_min_samples, metric='hamming')
    clusters_dbscan = dbscan.fit_predict(symptom_matrix)
    
    # Adicionar clusters ao DataFrame
    df_symptoms['cluster_dbscan'] = clusters_dbscan
    
    n_clusters = len(set(clusters_dbscan)) - (1 if -1 in clusters_dbscan else 0)
    n_noise = list(clusters_dbscan).count(-1)
    
    print(f"\n📊 Resultados DBSCAN:")
    print(f"   • Número de clusters encontrados: {n_clusters}")
    print(f"   • Registros classificados como ruído: {n_noise} ({n_noise/len(df_symptoms)*100:.2f}%)")
    print(f"   • Silhouette Score: {best_silhouette:.3f}")
    
    # Análise dos clusters (excluindo ruído)
    if n_clusters > 0:
        print(f"\n📊 Análise dos Clusters (DBSCAN):")
        cluster_counts = pd.Series(clusters_dbscan).value_counts().sort_index()
        for cluster_id, count in cluster_counts.items():
            if cluster_id != -1:  # Excluir ruído
                percentage = (count / len(df_symptoms)) * 100
                print(f"   Cluster {cluster_id}: {count:5,} registros ({percentage:5.2f}%)")
        
        # Identificar sintomas mais comuns em cada cluster
        print(f"\n🔍 Sintomas Característicos por Cluster:")
        for cluster_id in sorted(cluster_counts.index):
            if cluster_id != -1:  # Excluir ruído
                cluster_mask = clusters_dbscan == cluster_id
                cluster_symptoms = symptom_matrix[cluster_mask]
                symptom_freq = cluster_symptoms.sum(axis=0)
                
                # Top 5 sintomas mais frequentes neste cluster
                top_symptoms_idx = np.argsort(symptom_freq)[-5:][::-1]
                top_symptoms = [(frequent_symptoms[idx], symptom_freq[idx], 
                                (symptom_freq[idx] / cluster_symptoms.shape[0]) * 100) 
                               for idx in top_symptoms_idx if symptom_freq[idx] > 0]
                
                print(f"\n   Cluster {cluster_id} ({cluster_counts[cluster_id]:,} registros):")
                for symptom, count, pct in top_symptoms:
                    print(f"      • {symptom[:45]:45s} : {count:3d} ({pct:5.1f}%)")
    
    print("\n✅ DBSCAN concluído!")
else:
    print("❌ Não foi possível encontrar parâmetros adequados para DBSCAN")


## 9. Visualização e Análise Comparativa


In [ ]:
# Reduzir dimensionalidade para visualização (PCA)
print("📊 Reduzindo dimensionalidade com PCA para visualização...")
pca = PCA(n_components=2, random_state=42)
symptom_matrix_pca = pca.fit_transform(symptom_matrix_scaled)

print(f"   Variância explicada: {pca.explained_variance_ratio_.sum():.2%}")

# Criar visualizações comparativas
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# 1. K-Modes
if 'cluster_kmodes' in df_symptoms.columns:
    ax = axes[0, 0]
    scatter = ax.scatter(symptom_matrix_pca[:, 0], symptom_matrix_pca[:, 1], 
                        c=df_symptoms['cluster_kmodes'], cmap='viridis', 
                        alpha=0.6, s=20, edgecolors='black', linewidth=0.5)
    ax.set_title('K-Modes Clustering', fontsize=14, fontweight='bold')
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)', fontsize=11)
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)', fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax, label='Cluster')

# 2. K-Means
if 'cluster_kmeans' in df_symptoms.columns:
    ax = axes[0, 1]
    scatter = ax.scatter(symptom_matrix_pca[:, 0], symptom_matrix_pca[:, 1], 
                        c=df_symptoms['cluster_kmeans'], cmap='plasma', 
                        alpha=0.6, s=20, edgecolors='black', linewidth=0.5)
    ax.set_title('K-Means Clustering', fontsize=14, fontweight='bold')
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)', fontsize=11)
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)', fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax, label='Cluster')

# 3. Hierarchical
if 'cluster_hierarchical' in df_symptoms.columns:
    ax = axes[1, 0]
    scatter = ax.scatter(symptom_matrix_pca[:, 0], symptom_matrix_pca[:, 1], 
                        c=df_symptoms['cluster_hierarchical'], cmap='coolwarm', 
                        alpha=0.6, s=20, edgecolors='black', linewidth=0.5)
    ax.set_title('Hierarchical Clustering', fontsize=14, fontweight='bold')
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)', fontsize=11)
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)', fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax, label='Cluster')

# 4. DBSCAN
if 'cluster_dbscan' in df_symptoms.columns:
    ax = axes[1, 1]
    # Separar ruído dos clusters
    noise_mask = df_symptoms['cluster_dbscan'] == -1
    cluster_mask = ~noise_mask
    
    if cluster_mask.sum() > 0:
        scatter1 = ax.scatter(symptom_matrix_pca[cluster_mask, 0], 
                              symptom_matrix_pca[cluster_mask, 1], 
                              c=df_symptoms.loc[cluster_mask, 'cluster_dbscan'], 
                              cmap='Set3', alpha=0.6, s=20, 
                              edgecolors='black', linewidth=0.5)
        if noise_mask.sum() > 0:
            ax.scatter(symptom_matrix_pca[noise_mask, 0], 
                      symptom_matrix_pca[noise_mask, 1], 
                      c='red', marker='x', s=30, alpha=0.8, label='Noise')
        plt.colorbar(scatter1, ax=ax, label='Cluster')
    else:
        ax.scatter(symptom_matrix_pca[:, 0], symptom_matrix_pca[:, 1], 
                  c='red', marker='x', s=30, alpha=0.8)
    
    ax.set_title('DBSCAN Clustering', fontsize=14, fontweight='bold')
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)', fontsize=11)
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)', fontsize=11)
    ax.grid(True, alpha=0.3)
    if noise_mask.sum() > 0:
        ax.legend()

plt.tight_layout()
plt.show()

print("✅ Visualizações criadas!")


In [ ]:
# Comparação de distribuição de clusters
print("📊 Comparação de Distribuição de Clusters por Algoritmo:")
print("=" * 70)

algorithms = []
if 'cluster_kmodes' in df_symptoms.columns:
    algorithms.append(('K-Modes', 'cluster_kmodes'))
if 'cluster_kmeans' in df_symptoms.columns:
    algorithms.append(('K-Means', 'cluster_kmeans'))
if 'cluster_hierarchical' in df_symptoms.columns:
    algorithms.append(('Hierarchical', 'cluster_hierarchical'))
if 'cluster_dbscan' in df_symptoms.columns:
    algorithms.append(('DBSCAN', 'cluster_dbscan'))

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, (alg_name, col_name) in enumerate(algorithms):
    ax = axes[idx]
    cluster_counts = df_symptoms[col_name].value_counts().sort_index()
    
    # Para DBSCAN, destacar ruído
    if alg_name == 'DBSCAN' and -1 in cluster_counts.index:
        colors = ['red' if c == -1 else plt.cm.Set3(c) for c in cluster_counts.index]
        labels = [f'Noise' if c == -1 else f'Cluster {c}' for c in cluster_counts.index]
    else:
        colors = plt.cm.viridis(np.linspace(0, 1, len(cluster_counts)))
        labels = [f'Cluster {c}' for c in cluster_counts.index]
    
    bars = ax.bar(range(len(cluster_counts)), cluster_counts.values, color=colors, alpha=0.7, edgecolor='black')
    ax.set_xticks(range(len(cluster_counts)))
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_ylabel('Número de Registros', fontsize=11)
    ax.set_title(f'{alg_name} - Distribuição de Clusters', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Adicionar valores nas barras
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}',
                ha='center', va='bottom', fontsize=9)

# Ocultar eixos não utilizados
for idx in range(len(algorithms), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

# Tabela comparativa
print("\n📋 Tabela Comparativa de Algoritmos:")
comparison_data = []
for alg_name, col_name in algorithms:
    n_clusters = df_symptoms[col_name].nunique()
    if -1 in df_symptoms[col_name].values:
        n_clusters -= 1  # Excluir ruído da contagem
    
    # Calcular silhouette score
    if n_clusters > 1:
        mask = df_symptoms[col_name] != -1 if -1 in df_symptoms[col_name].values else slice(None)
        if mask.sum() > 1 and len(np.unique(df_symptoms.loc[mask, col_name])) > 1:
            sil_score = silhouette_score(symptom_matrix_scaled[mask], 
                                        df_symptoms.loc[mask, col_name])
        else:
            sil_score = np.nan
    else:
        sil_score = np.nan
    
    comparison_data.append({
        'Algoritmo': alg_name,
        'N_Clusters': n_clusters,
        'Silhouette_Score': f'{sil_score:.3f}' if not np.isnan(sil_score) else 'N/A'
    })

df_comparison = pd.DataFrame(comparison_data)
print(df_comparison.to_string(index=False))


## 10. Resumo e Conclusões


In [ ]:
print("📋 RESUMO E CONCLUSÕES")
print("=" * 70)

print(f"\n✅ ANÁLISE DE CLUSTERIZAÇÃO CONCLUÍDA:")
print(f"   • Total de registros analisados: {len(df_symptoms):,}")
print(f"   • Total de sintomas únicos considerados: {len(frequent_symptoms)}")
print(f"   • Algoritmos aplicados: {len(algorithms)}")

print(f"\n📊 PRINCIPAIS DESCOBERTAS:")
print(f"   • K-Modes: Específico para dados categóricos, identifica padrões baseados em moda")
print(f"   • K-Means: Funciona bem com dados binários após normalização")
print(f"   • Hierarchical: Permite visualizar hierarquia de clusters")
print(f"   • DBSCAN: Identifica clusters de forma arbitrária e detecta outliers")

print(f"\n💡 RECOMENDAÇÕES:")
print(f"   1. Comparar resultados com síndromes conhecidas (syndrome_id) se disponível")
print(f"   2. Validar clusters com especialistas em saúde")
print(f"   3. Considerar características temporais e geográficas para enriquecer análise")
print(f"   4. Ajustar parâmetros baseado em conhecimento de domínio")
print(f"   5. Explorar algoritmos adicionais como Gower Distance + clustering se disponível")

print(f"\n🎯 PRÓXIMOS PASSOS:")
print(f"   • Análise temporal dos clusters")
print(f"   • Análise geográfica dos clusters")
print(f"   • Validação com dados de síndromes conhecidas")
print(f"   • Feature engineering adicional")
print(f"   • Modelos preditivos baseados nos clusters")

print(f"\n✅ Notebook de clusterização concluído com sucesso!")
